# Product Evidence Platform — Azure ML Runner

This is the only supported notebook.

1. Copy `.env.example` to `.env` and replace every placeholder.
2. Place your private feature JSON in `inputs/private/<name>.json`.
3. Run `./scripts/azureml_startup.sh` from the Azure ML terminal.
4. Run the health-check cell below, then submit one product or a CSV batch.

The notebook is a thin API client. Search, scraping, browser control, image capture, vision reasoning, and artifact generation run inside the two containers.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import time
from pathlib import Path
from pprint import pprint
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

AGENT_URL = os.getenv("PRODUCT_AGENT_URL", "http://127.0.0.1:8788").rstrip("/")
POLL_SECONDS = 3
TERMINAL_STATUSES = {"COMPLETED", "REVIEW_REQUIRED", "FAILED"}

def api_json(method: str, path: str, payload: dict | None = None, timeout: int = 30) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{AGENT_URL}{path}",
        data=body,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Agent API returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(
            f"Cannot reach {AGENT_URL}. Run ./scripts/azureml_startup.sh in the terminal first."
        ) from exc

def check_health() -> dict:
    health = api_json("GET", "/health", timeout=15)
    if health.get("status") != "healthy":
        raise RuntimeError(f"Platform is not healthy: {json.dumps(health, indent=2)}")
    return health

def submit_product(product: dict, feature_set: str) -> str:
    response = api_json("POST", "/v1/jobs", {"product": product, "feature_set": feature_set})
    return response["job_id"]

def wait_for_job(job_id: str, poll_seconds: int = POLL_SECONDS) -> dict:
    while True:
        status = api_json("GET", f"/v1/jobs/{job_id}", timeout=15)
        print(
            f"{job_id}: {status['status']} | {status.get('stage', '')} | "
            f"{status.get('message', '')}"
        )
        if status["status"] in TERMINAL_STATUSES:
            if status["status"] == "FAILED":
                raise RuntimeError(status.get("error") or status.get("message") or "Job failed")
            return status
        time.sleep(poll_seconds)

def run_product(product: dict, feature_set: str) -> dict:
    job_id = submit_product(product, feature_set)
    wait_for_job(job_id)
    return api_json("GET", f"/v1/jobs/{job_id}/result", timeout=60)

health = check_health()
pprint(health)


## Single product

`FEATURE_SET` is the file name inside `inputs/private/` without the `.json` suffix.
The startup script creates a generic `example_features.json` only when the folder is empty. Replace it with the real private feature set for production work.


In [ ]:
FEATURE_SET = "example_features"

product = {
    "row_id": "ROW-001",
    "main_text": "Replace with the exact product identity text",
    "country_code": "CH",
    "retailer_name": None,
    "ean": None,
    "language_code": None,
}

result = run_product(product, FEATURE_SET)
pprint({
    "row_id": result.get("row_id"),
    "coding_ready": result.get("coding_ready"),
    "status": result.get("status"),
    "primary_url": result.get("primary_url"),
    "supplementary_urls": result.get("supplementary_urls"),
    "artifact_dir": result.get("artifact_dir"),
})


In [ ]:
# Inspect feature-level evidence and browser assets.
pprint(result.get("feature_evidence", []))
pprint(result.get("browser_evidence", []))


## Optional CSV batch

Expected columns:

`row_id, main_text, country_code, retailer_name, ean, language_code`

Blank optional values are converted to `None`. Results are written to `artifacts/notebook_batch_summary.csv`.


In [ ]:
BATCH_INPUT = Path("inputs/products.csv")
BATCH_OUTPUT = Path("artifacts/notebook_batch_summary.csv")

def blank_to_none(value: str | None) -> str | None:
    value = (value or "").strip()
    return value or None

def run_csv_batch(input_path: Path, output_path: Path, feature_set: str) -> list[dict]:
    if not input_path.is_file():
        raise FileNotFoundError(
            f"{input_path} does not exist. Create it or skip this optional batch cell."
        )

    summaries = []
    with input_path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))

    for index, row in enumerate(rows, start=1):
        product = {
            "row_id": blank_to_none(row.get("row_id")) or f"ROW-{index:05d}",
            "main_text": blank_to_none(row.get("main_text")),
            "country_code": blank_to_none(row.get("country_code")),
            "retailer_name": blank_to_none(row.get("retailer_name")),
            "ean": blank_to_none(row.get("ean")),
            "language_code": blank_to_none(row.get("language_code")),
        }
        if not product["main_text"] or not product["country_code"]:
            raise ValueError(f"Row {index} is missing main_text or country_code")

        item = run_product(product, feature_set)
        summaries.append({
            "row_id": product["row_id"],
            "coding_ready": item.get("coding_ready"),
            "status": item.get("status"),
            "primary_url": item.get("primary_url"),
            "supplementary_urls": "|".join(item.get("supplementary_urls") or []),
            "artifact_dir": item.get("artifact_dir"),
        })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summaries[0]) if summaries else ["row_id"])
        writer.writeheader()
        writer.writerows(summaries)
    return summaries

# Uncomment after creating inputs/products.csv.
# batch_results = run_csv_batch(BATCH_INPUT, BATCH_OUTPUT, FEATURE_SET)
# pprint(batch_results[:5])
